# core

> Run terminal processes with asyncio, stream output or read it in pages

`PtySession` runs a process in a pseudo-terminal. It supports two ways to read output from the same bounded buffer:

- A stream delivers bytes as they arrive, with recent scrollback on attachment. A websocket handler or renderer can use this.
- A cursor reads pages from a saved position. A tool can use this to retrieve new output between actions.

The buffer is a `Ring` with absolute byte offsets. Both kinds of reader can detect when buffer overflow has discarded output they haven't read.

In [ ]:
from ptymini.core import *
from nbdev.showdoc import show_doc


In [ ]:
import os, sys, time
from fastcore.test import test_eq

## The ring

`Ring.start` and `Ring.end` count bytes from the beginning of the terminal's output. Appending beyond `max_bytes` discards the oldest bytes without resetting these offsets.

`read_from(offset)` returns bytes from that position, capped by `max_bytes_out`. It also returns the next offset and the number of unread bytes lost to overflow.

In [ ]:
show_doc(Ring.read_from)

Check offsets, paging and lost-byte counts with an eight-byte buffer.

In [ ]:
r = Ring(max_bytes=8)
r.append(b'abcdef')
test_eq(r.read_from(0), (b'abcdef', 6, 0))
r.append(b'ghij')                      # 10 bytes total; the oldest 2 trim away
test_eq((r.start, r.end, len(r)), (2, 10, 8))
data, off, dropped = r.read_from(0)    # a reader at 0 lost the first 2 bytes
test_eq((data, off, dropped), (b'cdefghij', 10, 2))
page, off1, _ = r.read_from(2, max_bytes_out=3)   # paging: 3 bytes, then the rest
test_eq((page, off1), (b'cde', 5))
test_eq(r.read_from(off1)[0], b'fghij')
test_eq(r.read_from(off)[0], b'')      # caught up: empty, offset unmoved
r.read_from(off)

## PtySession

`PtySession` requires a running asyncio event loop. The Rust reader thread stores output in the ring and notifies the loop with `call_soon_threadsafe`. `wait_change` wakes on new output or EOF. `wait` returns the child's exit code.

Use `ptymini.bg` for synchronous calls.

In [ ]:
show_doc(PtySession)


In [ ]:
show_doc(PtySession.wait_change)

`terminate` sends SIGHUP, SIGCONT, SIGINT and SIGTERM in order until the child exits. This gives the shell and its foreground job a chance to clean up. With `force=True`, it also sends SIGKILL if needed. Exiting an `async with` block calls `terminate(force=True)`, including after an exception.

The signal sequence follows terminado's `PtyWithClients.terminate` (BSD-licensed, Copyright (c) Jupyter Development Team).


In [ ]:
show_doc(PtySession.terminate)


The examples use Bash without startup files and with a fixed prompt. Terminal reads can split output at different byte boundaries on each run. `read_until` accumulates bytes until the expected text appears or the child exits, with a timeout for waiting.

In [ ]:
BASH = ['bash', '--norc', '--noprofile', '-i']
BENV = dict(os.environ, PS1='$ ', TERM='dumb')

async def read_until(t, pat:bytes, offset:int=0, timeout=10.0):
    "Read from `offset` until `pat` appears in the accumulation (or EOF); returns (bytes, offset)."
    buf = b''
    end = time.monotonic() + timeout
    while pat not in buf:
        data, offset, _ = t.read_from(offset)
        buf += data
        if pat in buf or not (t.alive or data): break
        if not data: await t.wait_change(seen_end=offset, timeout=end - time.monotonic())
    return buf, offset


Start a shell and read a command's output using `read_from` and `wait_change`. The API accepts and returns bytes without decoding.


In [ ]:
t = PtySession(BASH, env=BENV)
t.write(b'echo hi $((6*7))\n')
out, off = await read_until(t, b'hi 42')
assert t.alive
out[-24:]


## Streams

`attach` returns an async iterator over the output. It yields retained scrollback, then new bytes, and ends at EOF. Each attachment tracks its own position. Stopping an iterator does not close the terminal or affect other readers.

If the reader falls behind the buffer, it resumes at the oldest retained byte. With `gaps=True`, the iterator first yields a `Gap` containing the number of lost bytes. A renderer can use this to detect incomplete output.

In [ ]:
show_doc(PtySession.attach)


A new attachment replays the command and output already in the ring. Reattachment after a refresh or disconnection can recover only the output the ring still retains.


In [ ]:
chunks = []
async for chunk in t.attach():
    chunks.append(chunk)
    if b'hi 42' in b''.join(chunks): break
assert b'echo hi' in b''.join(chunks)  # the command echo and its output both replayed from the ring
len(chunks)


The earlier stream read doesn't advance the `read_until` cursor. Check this by resizing the terminal and reading its reported size from that cursor.


In [ ]:
t.resize(40, 100)
t.write(b'stty size\n')
out, off = await read_until(t, b'40 100', offset=off)
b'40 100' in out


In [ ]:
assert await t.terminate(force=True)
code = await t.wait()
test_eq(t.alive, False)
final = [c async for c in t.attach()]       # a dead session's stream: full replay, then it ends
assert b'hi 42' in b''.join(final)
code                                        # negative: killed by that signal number


Check the exit code of a command that finishes on its own.

In [ ]:
async with PtySession([sys.executable, '-c', 'print("bye"); raise SystemExit(3)']) as p:
    test_eq(await p.wait(), 3)
    test_eq(p.alive, False)

Pause reading while the child produces more output than the ring can retain. The next read yields a `Gap`, followed by the remaining bytes.

In [ ]:
g = PtySession(['bash', '-c', 'printf a; sleep 0.3; printf "%0999d" 7'], buffer_bytes=64)
s = g.attach(gaps=True)
first = await anext(s)
test_eq(first, b'a')
await g.wait()                    # the flood happens while nobody reads the stream
nxt = await anext(s)
assert isinstance(nxt, Gap)
test_eq(int(nxt), 935)            # 1 byte read + 999 flooded - 64 retained
rest = b''.join([c async for c in s])
test_eq(len(rest), 64)
int(nxt), rest[-4:]

## The registry

`PtyRegistry` stores sessions by name. `create` returns an existing session with the requested name, or starts a new one. If no name is supplied, it chooses a number. `shutdown` terminates all sessions, as does exiting the registry's `async with` block.

Creation accepts an `argv` override and working directory. `env` replaces the inherited environment. `appendenv` adds or overrides values in that environment. Privilege changes belong to the host app, which can prefix `argv` with a command such as `sudo`.

Use `rc` to provide shell setup, such as prompt integration. The registry writes it to `.zshrc` in a private directory on the host. It substitutes `{rcfile}` and `{rcdir}` in `argv` and environment values:

- Bash accepts the file with `--rcfile {rcfile}`.
- Zsh reads the directory from `ZDOTDIR={rcdir}`.


In [ ]:
show_doc(PtyRegistry.create)

Create an auto-numbered session, retrieve it by name, then create one named `scratch`.

In [ ]:
terms = PtyRegistry(argv=BASH)
ta = await terms.create(env=BENV)
test_eq(ta.name, '1')
assert (await terms.create(name='1')) is ta          # existing name: same session back
tb = await terms.create(name='scratch', env=BENV)
[t.model()['name'] for t in terms.values()]


Define an alias in `rc` and check that the shell can use it.

In [ ]:
tr = await terms.create(argv=['bash', '--noprofile', '--rcfile', '{rcfile}', '-i'],
    rc="PS1='$ '\nalias hi='echo rc-worked'", env=dict(BENV))
tr.write(b'hi\n')
out, _ = await read_until(tr, b'rc-worked')
assert b'rc-worked' in out
tr.name

`cull_ready` lists sessions that have exceeded the inactivity timeout. `cull` terminates those sessions. Culling is disabled by default.

In [ ]:
test_eq(terms.cull_ready(), [])          # disabled by default
terms.cull_timeout = 3600
ta.last_activity -= 7200                 # stub the clock: ta has been idle two hours
test_eq(terms.cull_ready(), ['1'])
await terms.cull()
assert terms.get('1') is None and terms.get('scratch') is not None
len(terms)

The host app can run `cull_loop` as a task to check for inactive sessions periodically. Terminal reads and client writes count as activity.

In [ ]:
show_doc(cull_loop)


In [ ]:
await terms.shutdown()
test_eq(len(terms), 0)